# CysRank：结构信息驱动的半胱氨酸位点优选

本 Notebook 展示可复现的 **数据核对 → 模型训练与优化 → 分组评估 → 位点评分 → Top 3 导出** 流程。

项目融合 ProteinMPNN 预训练结构表征、局部序列上下文与四项生化特征，以轻量分类头实现 Cys 位点排序，并结合结构规则筛选候选。训练、评分和结果导出使用统一代码入口，支持来源追溯与后续实验反馈。

本轮已完成正例 SASA 特征更新后的数据整合、50 轮训练、模型重载核验和 28 项自动测试。Notebook 演示已有 Cys 位点筛选；项目另提供新增 Cys 的候选枚举及配套结构评分入口。

## 开始前

1. 在仓库根目录执行 `python -m pip install -r requirements.txt`。
2. 使用同一环境运行 `python -m jupyterlab`，打开本文件并选择该环境的 Python 内核。
3. 选择 **Restart Kernel and Run All Cells**。默认训练 50 轮，可在参数单元格改为 1 轮冒烟检查。

从仓库根目录或 notebooks 目录启动均可自动定位。若内核工作目录在仓库之外，先设置环境变量
`CYSRANK_ROOT` 指向移植后的仓库目录。后续数据、模型和输出路径全部基于识别出的仓库根目录。

当前版本 **sst-ranking-20260924-v1**：22 个分类位点 + 一条 Cys3 > Cys14 实验排序观察。100 个 MD 构象分摊一条观察的权重；SST 已纳入训练，不再作为独立验证。详细说明见 docs/SST_TRAINING.md。

In [1]:
from pathlib import Path
import os
import sys

hint = os.environ.get("CYSRANK_ROOT")
start = Path(hint).expanduser().resolve() if hint else Path.cwd().resolve()
roots = [start] if hint else [start, *start.parents]
ROOT = next((p for p in roots if all((p / name).is_file() for name in
    ("configs/train.json", "src/paths.py", "predict.py", "run.py"))), None)
if ROOT is None:
    raise RuntimeError("请从仓库内启动内核，或先设置 CYSRANK_ROOT 为仓库目录。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.paths import root_path, ROOT as CODE_ROOT
assert CODE_ROOT == ROOT, "当前内核加载了另一个项目，请重启内核。"
print("已定位仓库根目录；所有相对路径以此为基准。")

已定位仓库根目录；所有相对路径以此为基准。


In [2]:
import csv
import json
import html
import platform
import subprocess
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import torch
from IPython.display import display, HTML
from src import core, pipeline
from predict import export

def read_json(path):
    return Path(path).read_text(encoding="utf-8-sig")

def table(rows):
    if not rows:
        display(HTML("<p>没有符合条件的记录。</p>"))
        return
    columns = list(rows[0])
    def cell(value):
        return html.escape(str(value))
    header = "".join("<th>" + cell(k) + "</th>" for k in columns)
    body = "".join("<tr>" + "".join("<td>" + cell(r.get(k, "")) + "</td>" for k in columns) + "</tr>" for r in rows)
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"))

table([{"Python": platform.python_version(), "PyTorch": str(torch.__version__),
        "NumPy": np.__version__, "CUDA可用": torch.cuda.is_available()}])

Python,PyTorch,NumPy,CUDA可用
3.13.5,2.11.0+cu128,2.5.2,True


## 1. 参数和输出位置

输入为项目随附的无标签特征文件；`feature_mask` 只选择特征已准备的 Cys。
训练读取 configs/train.json 指定的正负记录和母蛋白分组信息，与推理输入分别管理。
每次运行此单元格都会创建新的运行编号，重复 Run All 不覆盖正式模型和结果。
演示赛道必须在正式提交前替换为真实赛道。

In [3]:
EPOCHS = 50
TOP_K = 3
TRACK = "未指定（演示）"
INPUT = "data/examples/input.jsonl"
CHECKPOINT = "models/site_predictor.pt"
assert isinstance(EPOCHS, int) and EPOCHS > 0
assert isinstance(TOP_K, int) and TOP_K > 0
config = json.loads(read_json(root_path("configs/train.json")))
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid4().hex[:8]
RUN = root_path("logs/notebook") / run_id
RUN.mkdir(parents=True, exist_ok=False)
print("本次输出：", RUN.relative_to(ROOT).as_posix())
print("轮数：", EPOCHS, "；随机种子：", config["seed"])

本次输出： logs/notebook/20260924T174028_358cc906
轮数： 50 ；随机种子： 20260916


## 2. 数据更新与质量控制

本轮纳入更新后的正例 SASA 特征，训练数据包括 **8 条正例、11 条负例，共 22 个监督位点（8 正、14 负）**。沿用既有数据清洗决定隔离 `VNRRPCFSALT`，保留原始来稿、排除记录和文件哈希，确保每次更新均可追溯。

原始 ca_label 用于标记已测试位点，训练结局依据正负文件来源构造；未标注 Cys 保持未知。外部 248 条候选数据与训练监督分开管理，避免将未验证记录误用为实验阴性。

此次更新范围为正例特征和样本组成，负例保持原版。结构仍以提供的片段构象为依据，本轮不涉及游离短肽构象重建。下方直接读取训练输入并核对来源哈希，展示实际数据规模与版本记录。

In [4]:
training = pipeline.prepare(root_path(config["source"]), root_path(config["parents"]))
examples = core.load_jsonl(root_path(INPUT))
for row in examples:
    core.validate(row)
assert len({r["record_id"] for r in examples}) == len(examples)
assert all("ca_label" not in r for r in examples)
stats = {
    "训练记录": len(training),
    "监督位点": sum(sum(r["label_mask"]) for r in training),
    "正位点": sum(sum(r["ca_label"]) for r in training),
    "负位点": sum(sum(r["confirmed_negative_mask"]) for r in training),
    "推断母蛋白组": len({r["parent_group"] for r in training}),
    "推理示例记录": len(examples),
}
table([stats])
table([{ "记录": r["record_id"], "序列": r["seq"], "长度": len(r["seq"]),
         "可评分Cys数": sum(aa == "C" and bool(m) for aa, m in zip(r["seq"], r["feature_mask"]))}
       for r in examples[:5]])
revision = json.loads(read_json(root_path("data/provenance/revision_20260924.json")))
for path, digest in revision["active_file_sha256"].items():
    assert core.sha(root_path(path)) == digest, path
assert not any(r["seq"] == "VNRRPCFSALT" for r in training)
print("本轮来源哈希核对通过；排除记录：", revision["excluded_positive"])
print("与 2MI1 重叠的训练序列：", [r["seq"] for r in training if r["seq"] in "AGCKNFFWKTFTSC"])
ordering = json.loads(read_json(root_path(config["ranking"])))
print("新增排序观察：", ordering["observation_id"], "；构象数：", len(ordering["samples"]), "；实验观察数：1")

训练记录,监督位点,正位点,负位点,推断母蛋白组,推理示例记录
19,22,8,14,5,19


记录,序列,长度,可评分Cys数
positive_001,ECG,3,1
positive_002,ALYLVCGERG,10,1
positive_003,AGCKNFFW,8,1
positive_004,KTFTSC,6,1
positive_005,VHKECCHGDLL,11,1


本轮来源哈希核对通过；排除记录： {'sequence': 'VNRRPCFSALT', 'source_line_1based': 8, 'reason': 'Continue user-authorized prior exclusion for conflict with previous-Pro output filter; not relabeled negative'}
与 2MI1 重叠的训练序列： ['AGCKNFFW', 'KTFTSC']
新增排序观察： sst_cys3_gt_cys14_user_report_20260924 ；构象数： 100 ；实验观察数：1


## 3. 训练、优化与筛选

调用正式 run.py，使用内核的同一解释器。冻结编码器并用 AdamW 优化分类头；
保持固定轮次和阈值，不根据留出结果选模型。本次训练写入独立目录，**不会替换 models/site_predictor.pt**。
完整 stdout/stderr 保存到运行目录，失败时抛出异常停止流程。

In [5]:
workflow_dir = RUN / "workflow"
command = [sys.executable, str(root_path("run.py")), "--output",
           workflow_dir.relative_to(ROOT).as_posix(), "--epochs", str(EPOCHS), "--track", TRACK]
completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True,
                           encoding="utf-8", errors="replace")
(RUN / "workflow.log").write_text(completed.stdout + completed.stderr, encoding="utf-8")
if completed.returncode:
    raise RuntimeError("训练流程失败，请查看 " + (RUN / "workflow.log").relative_to(ROOT).as_posix()
                       + "\n" + completed.stderr[-3000:])
summary = json.loads(read_json(workflow_dir / "training/summary.json"))
timing = json.loads(read_json(workflow_dir / "timing.json"))
table([{"训练轮数": EPOCHS, "耗时秒": round(timing["elapsed_seconds"], 2),
        "训练位点": summary["training_sites"], "模型重新加载通过": summary["reload_verified"]}])

训练轮数,耗时秒,训练位点,模型重新加载通过
50,5.72,22,True


## 当前模型训练结果与分组诊断

以下直接读取本次复现日志。原分类监督保留，新增一条 SST 排序观察；训练排序符合仅说明拟合已知实验。分组留出时整条 SST 及相关片段同时移出训练。

In [6]:
table([{"范围": "训练回代（原始）", **summary["training"]["raw"]},
       {"范围": "训练回代（加过滤）", **summary["training"]["filtered"]}])
diagnostic = summary["diagnostics"]["full"]
table([{"留出母蛋白": f["parent"], "状态": f["status"],
        "训练位点": f["train_n"], "留出位点": f["held_n"],
        "召回率": f.get("raw", {}).get("recall", "不适用")}
       for f in diagnostic["folds"]])
print("合并留出指标：")
display(diagnostic["pooled"])
loss_log = json.loads(read_json(workflow_dir / "training/training_log.json"))
print("损失日志首尾记录：")
display(loss_log[:2] + loss_log[-2:] if len(loss_log) > 4 else loss_log)
table([{"消融": name, "可评估位点": item["pooled"]["n"], "命中": item["pooled"]["tp"],
        "诊断Recall": item["pooled"]["recall"]} for name, item in summary["diagnostics"].items()])
print("SST 排序训练回代（非独立验证）：")
display(summary["ranking_training"])
print("整组留出后的 SST 诊断：")
display([f["heldout_ordering"] for f in summary["diagnostics"]["full"]["folds"] if "heldout_ordering" in f])

范围,n,positive_sites,negative_sites,tp,fp,tn,fn,accuracy,precision,recall,specificity,f1,average_precision,threshold,note
训练回代（原始）,22,8,14,8,0,14,0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,"Average precision is threshold-grouped AP, not trapezoidal PR-AUC."
训练回代（加过滤）,22,8,14,8,0,14,0,1.0,1.0,1.0,1.0,1.0,None,0.5,"Average precision is threshold-grouped AP, not trapezoidal PR-AUC."


留出母蛋白,状态,训练位点,留出位点,召回率
old3_positive_03,evaluated,21,1,1.0
old3_positive_04,evaluated,21,1,1.0
old3_positive_05,evaluated,20,2,0.0
old3_positive_06,skipped_single_class_training,5,17,不适用
old3_positive_07,evaluated,21,1,1.0


合并留出指标：


{'n': 5,
 'positive_sites': 5,
 'negative_sites': 0,
 'tp': 3,
 'fp': 0,
 'tn': 0,
 'fn': 2,
 'accuracy': None,
 'precision': None,
 'recall': 0.6,
 'specificity': None,
 'f1': None,
 'average_precision': None,
 'threshold': 0.5,
 'note': 'Single-class sets cannot estimate both sensitivity and specificity; AP omitted.'}

损失日志首尾记录：


...

消融,可评估位点,命中,诊断Recall
full,5,3,0.6
biochemical_only,5,0,0.0
without_biochemical,5,3,0.6


SST 排序训练回代（非独立验证）：


{'conformers': 100,
 'preferred_higher': 99,
 'preferred_mean': 0.9836229681968689,
 'other_mean': 0.7697407603263855,
 'scope': 'ONE experimental ordering; conformers are correlated augmentation, not independent tests'}

整组留出后的 SST 诊断：


[{'conformers': 100,
  'preferred_higher': 26,
  'preferred_mean': 0.4167950451374054,
  'other_mean': 0.5237771272659302,
  'scope': 'ONE experimental ordering; conformers are correlated augmentation, not independent tests'}]

## 5. 候选优选与 Top 3 导出

本单元格加载随附正式模型，按统一接口完成评分、规则过滤与全局排序。分数 ≥ 0.5 且通过二硫键、前位 Pro 过滤的位点进入候选清单；同分时按记录编号和位点排序，保证输出可复现。

结果同时保存 Top 3 CSV、全位点评分及筛选原因，支持逐条复核与后续验证。本例采用训练来源数据展示流程；分数用于候选优先级排序。提供匹配的全原子结构时可附带结构文件引用。

In [7]:
# 独立编号也允许只重复执行本单元格，而不覆盖前次导出。
candidate_path = RUN / ("delivered_model_" + uuid4().hex[:8] + ".csv")
export(root_path(INPUT), root_path(CHECKPOINT), candidate_path, TRACK, TOP_K)
with candidate_path.open(encoding="utf-8", newline="") as stream:
    candidates = list(csv.DictReader(stream))
columns = ["candidate_id", "sequence", "position_1based", "prediction_score", "rank",
           "is_disulfide", "sasa_A2", "prev_is_pro", "nearby_atoms_5A"]
table([{k: r[k] for k in columns} for r in candidates])
print("候选 CSV：", candidate_path.relative_to(ROOT).as_posix())
print("全位点审计：", candidate_path.with_suffix(".sites.jsonl").relative_to(ROOT).as_posix())

D:\CysRank-GCLSC\src\protein_mpnn_utils.py:799: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Cross.cpp:67.)
  n_2 = F.normalize(torch.cross(u_2, u_1), dim=-1)


Scored 22 existing Cys sites; scores are not calibrated efficiencies.
Exported 3 candidates to delivered_model_a8a18fc8.csv


candidate_id,sequence,position_1based,prediction_score,rank,is_disulfide,sasa_A2,prev_is_pro,nearby_atoms_5A
positive_001_C2,ECG,2,0.99440467,1,0,141.8,0,8
positive_008_C6,SKYPNCAYKTT,6,0.99161106,2,0,8.57,0,33
positive_003_C3,AGCKNFFW,3,0.98681730,3,0,19.56,0,15


候选 CSV： logs/notebook/20260924T174028_358cc906/delivered_model_a8a18fc8.csv
全位点审计： logs/notebook/20260924T174028_358cc906/delivered_model_a8a18fc8.sites.jsonl


## 6. 复现核验

按记录编号和位点匹配本轮正式预测，不依赖文件行顺序。对默认数据和随附权重，
分数差应在浮点误差范围内；若换用自有输入或模型，则不与原始样例强行比较。
新训练结果也与本轮正式训练摘要对照。改为少量轮次后，不要求拟合结果与 50 轮模型一致。

In [8]:
current = core.load_jsonl(candidate_path.with_suffix(".sites.jsonl"))
report = {"epochs": EPOCHS, "seed": config["seed"], "exported_candidates": len(candidates),
          "scored_sites": len(current), "run_directory": RUN.relative_to(ROOT).as_posix()}
if INPUT == "data/examples/input.jsonl" and CHECKPOINT == "models/site_predictor.pt":
    historical = core.load_jsonl(root_path("logs/retrain_sst_20260924/final_model/reload_predictions.jsonl"))
    key = lambda r: (r["record_id"], r["position_1based"])
    expected = {key(r): r for r in historical}
    assert {key(r) for r in current} == set(expected)
    delta = max((abs(r["raw_score"] - expected[key(r)]["raw_score"]) for r in current), default=0.0)
    assert delta <= 1e-6, f"分数差异超出容差：{delta}"
    assert all(r["predicted_positive"] == expected[key(r)]["predicted_positive"] for r in current)
    report["max_delivered_model_score_difference"] = delta
original_summary = json.loads(read_json(root_path("logs/retrain_sst_20260924/summary.json")))
report["training_metrics_match_original"] = summary["training"] == original_summary["training"]
report["grouped_diagnostics_match_original"] = summary["diagnostics"] == original_summary["diagnostics"]
core.write_json(RUN / "notebook_verification.json", report)
display(report)
print("新训练模型的候选示例：")
with (workflow_dir / "results.csv").open(encoding="utf-8", newline="") as stream:
    retrained_candidates = list(csv.DictReader(stream))
table([{k: r[k] for k in columns} for r in retrained_candidates])
if EPOCHS == 50:
    assert report["training_metrics_match_original"]
    assert report["grouped_diagnostics_match_original"]

{'epochs': 50,
 'seed': 20260916,
 'exported_candidates': 3,
 'scored_sites': 22,
 'run_directory': 'logs/notebook/20260924T174028_358cc906',
 'max_delivered_model_score_difference': 0.0,
 'training_metrics_match_original': True,
 'grouped_diagnostics_match_original': True}

新训练模型的候选示例：


candidate_id,sequence,position_1based,prediction_score,rank,is_disulfide,sasa_A2,prev_is_pro,nearby_atoms_5A
positive_001_C2,ECG,2,0.99440467,1,0,141.8,0,8
positive_008_C6,SKYPNCAYKTT,6,0.99161106,2,0,8.57,0,33
positive_003_C3,AGCKNFFW,3,0.98681730,3,0,19.56,0,15


## 交付成果与后续拓展

本轮已形成配套的训练数据快照、正式模型权重、训练与评估日志、Top 3 示例，以及可从头执行的 Notebook。28 项自动测试覆盖监督语义、反馈隔离、模型输入、结构评分、插入设计和能垒接口；正式模型重载与本 Notebook 的复现结果已核对一致。

每次运行的 logs/notebook/<运行编号>/ 保存训练权重、配置、分组消融、候选结果和 notebook_verification.json，便于复核、展示与后续迭代。旧版数据和模型已归档。

项目已支持新 Cys 候选枚举、外部配套结构评分和实验反馈数据接入。后续重点是接入候选结构预测，并扩展独立游离短肽实验验证。

**使用范围：** 当前示例用于方法展示与候选初筛，分数不等于环化效率。2MI1 与本轮训练片段存在来源重叠，不列为独立验证；详细数据及评估边界见 data/README.md 和 models/MODEL_CARD.md。